# RDF Extractions from all data zeroshot

In [ ]:
import base64
import json
import os
from pathlib import Path
import requests
from PIL import Image

# 1. CONFIGURATION
# It's recommended to load the API key from an environment variable for security.
def load_api_key(name, fallback_names=()):
    import os
    import sys
    from pathlib import Path

    candidates = (name, *fallback_names)
    repo_root = next(
        (
            candidate
            for candidate in (Path.cwd(), *Path.cwd().parents)
            if (candidate / "common" / "__init__.py").exists()
        ),
        None,
    )

    get_api_key = None
    if repo_root is not None:
        if str(repo_root) not in sys.path:
            sys.path.insert(0, str(repo_root))
        try:
            from common import get_api_key as shared_get_api_key
        except ImportError:
            pass
        else:
            get_api_key = shared_get_api_key

    if get_api_key is not None:
        for candidate in candidates:
            try:
                return get_api_key(candidate)
            except KeyError:
                pass

    for candidate in candidates:
        value = os.getenv(candidate, "").strip()
        if value:
            return value

    joined = ", ".join(candidates)
    raise RuntimeError(
        f"Missing API key. Set one of [{joined}] in the environment"
        " or the top-level config file."
    )

GEMINI_API_KEY = load_api_key("GEMINI_API_KEY")

MODEL = "gemini-2.5-flash"  # Using a more recent model, but you can change it back
ENDPOINT = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent?key={GEMINI_API_KEY}"

# --- NEW: Specify your dataset directory ---
DATASET_DIR = "/content/drive/MyDrive/Research/Wageningen University /datasets/data"  # <--- IMPORTANT: Change this to the path of your dataset
OUTPUT_FILENAME = "rdf_extractions.json"


# 2. FUNCTIONS (Mostly unchanged, with minor improvements)

def encode_image_to_base64(image_path: str) -> str:
    """Encodes an image to a base64 string."""
    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB")
            buffer = Path(image_path).read_bytes()
            return base64.b64encode(buffer).decode("utf-8")
    except FileNotFoundError:
        print(f"Error: Image file not found at {image_path}")
        return None
    except Exception as e:
        print(f"An error occurred while processing the image at {image_path}: {e}")
        return None

def build_request(image_b64: str, prompt: str) -> dict:
    """Builds the JSON payload for the Gemini API request."""
    return {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {"inlinedata": {"mimeType": "image/jpeg", "data": image_b64}},
                    {"text": prompt},
                ],
            }
        ],
        "generationConfig": {
            "temperature": 0.1,
            "topP": 0.9,
            "maxOutputTokens": 8192,
        },
    }

def call_gemini(payload: dict) -> str:
    """Calls the Gemini API and returns the extracted RDF text."""
    headers = {"Content-Type": "application/json"}
    try:
        response = requests.post(ENDPOINT, headers=headers, data=json.dumps(payload))
        response.raise_for_status()
        result = response.json()

        if "candidates" in result:
            extracted_rdf = result["candidates"][0]["content"]["parts"][0]["text"]
            # Clean up the response to get just the Turtle code
            if extracted_rdf.strip().startswith("```turtle"):
                extracted_rdf = extracted_rdf.strip()[len("```turtle"):-len("```")].strip()
            return extracted_rdf
        return None
    except requests.exceptions.HTTPError as e:
        print(f"An HTTP error occurred: {e}\nResponse body: {e.response.text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None


# 3. MAIN EXECUTION (This is where the main changes are)

def main():
    """
    Main function to iterate through image datasets, extract RDF,
    and save the combined results.
    """
    # This is the same detailed prompt you provided.
    prompt_text = """
    
    You are a multimodal knowledge-extraction agent.

    ### Objective
    From a single diagram, flowchart, chart, or table, extract the concepts (nodes) and relationships (edges) that are explicitly present and output **only** RDF triples in valid Turtle using **only** the SKOS and `she:` namespaces.

    ### Input Assumptions
    - You receive one image at a time.
    - Treat visible text literally (case, punctuation, numbers).
    - If something is unclear or not visible, **omit it** rather than guessing.

    ### Namespaces (always emit first)
    ```turtle
    @prefix she:  <https://soilwise-he.github.io/soil-health#> .
    @prefix skos: <http://www.w3.org/2004/02/skos/core#> .
    ```

    ### Nodes (Concepts)
    - Mint a URI in the `she:` namespace using **PascalCase** derived from the exact label text (normalize as below).
    - Declare each node `a skos:Concept`.
    - Add `skos:prefLabel` with the **exact text from the image, but in all lowercase**.
    - If the image provides a definition for a node, add `skos:definition` with that text in lowercase.
    - **Deduplicate**: if the same label appears multiple times, create one concept and reuse its URI.

    **URI normalization (deterministic)**
    - Start from the label’s visible text.
    - Remove leading/trailing whitespace.
    - Remove quotes and punctuation except hyphens and slashes.
    - Split on whitespace, hyphens, slashes, and underscores; capitalize each token; concatenate (PascalCase).
    - Remove diacritics; keep ASCII letters and digits only.
    - If the result starts with a digit, prefix with a capitalized word (e.g., `Concept`).
    - Examples of mapping (for internal guidance only; do not output these):
    - "microbial biomass c (bacteria, fungi)" → `she:MicrobialBiomassC`
    - "inert organic matter" → `she:InertOrganicMatter`
    - "water storage / quality" → `she:WaterStorageQuality`

    ### Edges (Relationships)
    - Use `skos:narrower` / `skos:broader` for **hierarchies** (e.g., containers, bullets, tree levels, parent→child boxes).
    - For **non-hierarchical relations** explicitly shown (arrows, connectors, labeled links), create a **custom property** in `she:` using **camelCase** that clearly states the relation’s meaning (e.g., `she:measures`, `she:affects`, `she:involves`, `she:hasCriterion`, `she:alsoKnownAs`).
    - Prefer **object properties** (linking concepts) when both ends are concepts.
    - For **data values** that are not concepts (numbers, units, percentages, short attributes), attach **plain string literals** via a clear camelCase property in `she:` (e.g., `she:hasApproximateShareOfTotalSOM "5–25%"`).
    - If an edge has a **direction** (arrow), respect it in the triple `subject predicate object`.

    ### What to Extract
    - Box titles, list items, table headers/rows as concepts.
    - Labeled arrows/connectors as custom properties (use the label text to name the property when meaningful; otherwise pick the clearest verb).
    - Unlabeled arrows: use a generic but precise verb (e.g., `she:leadsTo`, `she:resultsIn`, `she:dependsOn`) chosen to best match the diagram semantics.
    - Synonyms/aliases explicitly shown: use `she:alsoKnownAs`.
    - Grouping/containment (e.g., a frame with items inside): model as `skos:narrower` from the group to each item.

    ### Output Requirements (strict)
    - **Output only valid Turtle. No explanations, no headings, no comments.**
    - Emit the two prefixes first (exactly as above).
    - One triple per line, each ending with a period.
    - Group all triples for the **same subject** together using semicolons.
    - Use only `she:` and `skos:` namespaces; **no other prefixes** (e.g., no `rdf:`, `rdfs:`, `xsd:`).
    - Use **plain string literals** (double-quoted) for all literal values; do not add datatypes or language tags.
    - Do not emit blank nodes.
    - Do not invent content that is not visible.

    ### Post-processing Checklist (must hold before you answer)
    - [ ] Prefix block present and exact.
    - [ ] Every subject URI is `she:PascalCase`.
    - [ ] Every concept has `a skos:Concept` and a lowercase `skos:prefLabel`.
    - [ ] Hierarchies use `skos:narrower` / `skos:broader` consistently (avoid cycles).
    - [ ] Custom properties are `she:camelCase` and semantically clear.
    - [ ] No undeclared prefixes; no commentary.
    - [ ] One triple per line; semicolons used to group by subject; every line ends with a period.
    - [ ] No duplicates of the same triple.

    ### If Nothing Extractable
    - Still output the prefix block and nothing else.



    """

    all_rdf_data = []
    # --- NEW: Loop through the dataset directory ---
    for root, dirs, files in os.walk(DATASET_DIR):
        for file in files:
            # --- NEW: Check for image file extensions ---
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(root, file)
                print(f"Processing image: {img_path}")

                image_b64 = encode_image_to_base64(img_path)

                if image_b64:
                    request_body = build_request(image_b64, prompt_text)
                    print("Sending request to Gemini API...")
                    extracted_rdf = call_gemini(request_body)

                    if extracted_rdf:
                        print(f"Successfully extracted RDF from {img_path}")
                        # --- NEW: Append result to our list ---
                        all_rdf_data.append({
                            "source_image": img_path,
                            "rdf_graph_turtle": extracted_rdf
                        })
                    else:
                        print(f"Failed to get a valid response for {img_path}")
                print("-" * 20) # Separator for clarity

    # --- NEW: Save all collected RDF data to a single JSON file ---
    if all_rdf_data:
        output_data = {"dataset": all_rdf_data}
        with open(OUTPUT_FILENAME, "w", encoding="utf-8") as f:
            json.dump(output_data, f, indent=4)
        print(f"\n--- Successfully saved all RDF extractions to {OUTPUT_FILENAME} ---")
    else:
        print("No RDF data was extracted from any images.")

if __name__ == "__main__":
    main()